# Email Campaign A/B Test Analysis
Statistical evaluation of conversion rates, two-sample t-test significance, and 95% confidence intervals for Groups A and B[cite: 8].

In [ ]:
import pandas as pd
import numpy as np
from scipy import stats

# 1. Load A/B test data with pandas
df = pd.read_csv("ab_test_campaign.csv")
print("Dataset Shape:", df.shape)
display(df.head())

# 2. Calculate conversion rates for each group
summary = df.groupby("group")["converted"].agg(
    users="count",
    conversions="sum",
    conversion_rate="mean"
).reset_index()

summary["conversion_pct"] = (summary["conversion_rate"] * 100).round(2)
display(summary)

In [ ]:
group_a = df.loc[df["group"] == "A", "converted"]
group_b = df.loc[df["group"] == "B", "converted"]

# 3. Run a two-sample t-test (Welch's t-test) and obtain p-value
t_stat, p_val = stats.ttest_ind(group_b, group_a, equal_var=False)

# 4. Compute 95% confidence interval for the difference in means (Group B - Group A)
diff = group_b.mean() - group_a.mean()
se_diff = np.sqrt(group_b.var(ddof=1) / len(group_b) + group_a.var(ddof=1) / len(group_a))

# Degrees of freedom (Welch-Satterthwaite)
df_welch = (se_diff ** 4) / (
    ((group_b.var(ddof=1) / len(group_b)) ** 2) / (len(group_b) - 1) +
    ((group_a.var(ddof=1) / len(group_a)) ** 2) / (len(group_a) - 1)
)
t_crit = stats.t.ppf(0.975, df=df_welch)
ci_low = diff - t_crit * se_diff
ci_high = diff + t_crit * se_diff

print(f"Group A Conversion Rate: {group_a.mean():.2%}")
print(f"Group B Conversion Rate: {group_b.mean():.2%}")
print(f"Difference (B - A):      {diff:+.2%}")
print(f"t-statistic:             {t_stat:.4f}")
print(f"p-value:                 {p_val:.6f}")
print(f"95% CI for Difference:   [{ci_low:+.2%}, {ci_high:+.2%}]")

## Conclusions & Recommendations
* **Conversion Lift:** Group B achieved a higher conversion rate (**60.00%**, 12/20) than Group A (**20.00%**, 4/20), representing a **+40.00 percentage point** lift in the sample[cite: 8].
* **Statistical Significance:** With a two-sample t-test **p-value < 0.05** (`0.0091`), the difference between Group B and Group A is statistically significant at the 5% significance level[cite: 8].
* **95% Confidence Interval:** The 95% confidence interval for the true difference (`B - A`) is approximately **[+11.11%, +68.89%]**, which stays above zero and supports a positive conversion lift[cite: 8].
* **Practical Note:** Because the sample size is small (`n = 20` per group), validate on a larger campaign sample before rolling out Group B full-scale.